<a href="https://colab.research.google.com/github/andaba2101/Proyecto_Final_RappiPlus_Data_Analyst/blob/main/S12_Estudiante_Proyecto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv")
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv")
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")

In [ ]:
# Creación de formula generalizable
def resumen_calidad(df):

    print('----- INFORMACIÓN GENERAL -----')

    print('\nNúmero de filas:', df.shape[0])
    print('Número de columnas:', df.shape[1])

    print('\n----- NOMBRES DE COLUMNAS -----')
    print(df.columns)

    print('\n----- TIPOS DE DATOS -----')
    print(df.info())

    print('\n----- VALORES FALTANTES -----')
    nulos = df.isna().sum()
    print(nulos[nulos > 0])

    print('\n----- FILAS DUPLICADAS COMPLETAMENTE -----')

    # duplicated() revisa todas las columnas de una fila
    # y cuenta cuántas filas son repetidas completamente
    print('Cantidad de filas duplicadas:', df.duplicated().sum())


    print('\n----- DUPLICADOS EN ID_PEDIDO -----')

    # Verificar primero si la columna id_pedido existe en el DataFrame
    if 'id_pedido' in df.columns:

        # duplicated() sobre id_pedido cuenta cuántos IDs aparecen repetidos
        print(
            'Cantidad de id_pedido repetidos:',
            df['id_pedido'].duplicated().sum()
        )

        # nunique() cuenta IDs distintos, sin repetirlos
        print(
            'Cantidad de id_pedido únicos:',
            df['id_pedido'].nunique()
        )

    # Si el DataFrame no tiene la columna id_pedido,
    # mostrar un mensaje en vez de producir un error
    else:
        print('Este DataFrame no contiene la columna id_pedido.')

    print('\n----- PRIMERAS CINCO FILAS -----')
    print(df.head())

    print('\n----- VALORES UNICOS EN CADA COLUMNA -----')
    print(df.nunique())

    # Bloque nuevo: detalle de columnas categóricas (texto)
    print('\n----- DETALLE DE COLUMNAS CATEGÓRICAS -----')

    # Seleccionar columnas de texto (object o string)
    columnas_texto = df.select_dtypes(include=['object', 'string']).columns

    # Recorrer cada columna de texto
    for columna in columnas_texto:
        print(f'\nColumna: {columna}')

        # Mostrar valores únicos y su frecuencia
        # value_counts cuenta cuántas veces aparece cada valor
        # dropna=False incluye los valores nulos en el conteo
        print(df[columna].value_counts(dropna=False))

# Validación de datos de orders
resumen_calidad(orders)


----- INFORMACIÓN GENERAL -----

Número de filas: 25100
Número de columnas: 12

----- NOMBRES DE COLUMNAS -----
Index(['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo',
       'fuente_referencia', 'nombre_producto', 'categoria_producto',
       'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total'],
      dtype='object')

----- TIPOS DE DATOS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad      

In [ ]:
# Validación de datos de catalog
resumen_calidad(catalog)

----- INFORMACIÓN GENERAL -----

Número de filas: 7
Número de columnas: 4

----- NOMBRES DE COLUMNAS -----
Index(['nombre_producto', 'categoria_producto', 'costo_unitario', 'proveedor'], dtype='object')

----- TIPOS DE DATOS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
None

----- VALORES FALTANTES -----
Series([], dtype: int64)

----- FILAS DUPLICADAS COMPLETAMENTE -----
Cantidad de filas duplicadas: 0

----- DUPLICADOS EN ID_PEDIDO -----
Este DataFrame no contiene la columna id_pedido.

----- PRIMERAS CINCO FILAS -----
        nombre_producto categoria_producto  costo_unitario  \

In [ ]:
resumen_calidad(marketing)

----- INFORMACIÓN GENERAL -----

Número de filas: 1620
Número de columnas: 5

----- NOMBRES DE COLUMNAS -----
Index(['fecha', 'pais', 'id_campaña', 'canal', 'gasto'], dtype='object')

----- TIPOS DE DATOS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB
None

----- VALORES FALTANTES -----
canal    101
dtype: int64

----- FILAS DUPLICADAS COMPLETAMENTE -----
Cantidad de filas duplicadas: 0

----- DUPLICADOS EN ID_PEDIDO -----
Este DataFrame no contiene la columna id_pedido.

----- PRIMERAS CINCO FILAS -----
        fecha      pais            id_campaña        canal    gasto
0  2025-01-01    

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# Categorización de Columnas y dataframes en listas

# Columnas Numericas de cada dataframe
col_num_orders = orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']]
col_num_catalog = catalog[['costo_unitario']]
col_num_marketing = marketing[['gasto']]

# Columnas Categoricas de cada dataframe

col_cat_orders = orders[['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']]
col_cat_catalog = catalog[['nombre_producto', 'categoria_producto', 'proveedor']]
col_cat_marketing = marketing[['pais', 'canal']]



In [ ]:
# Validar y convertir fechas al formato correcto
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

In [ ]:
# Revisar variables numéricas (sin negativos o ceros inválidos)

In [ ]:
# ANALISIS DE COLUMNAS NUMERICAS DE CADA DATAFRAME

def valores_nulos(df, columnas):
    for col in columnas:
        print(f"Cantidad de valores nulos en columna {col}")
        print(df[col].isna().sum())
        print(f"Proporción de valores nulos en columna {col}")
        print(df[col].isna().mean())
        print("_____________________________________")
    return

valores_nulos(orders, col_num_orders)


Cantidad de valores nulos en columna cantidad
50
Proporción de valores nulos en columna cantidad
0.00199203187250996
_____________________________________
Cantidad de valores nulos en columna precio_unitario
50
Proporción de valores nulos en columna precio_unitario
0.00199203187250996
_____________________________________
Cantidad de valores nulos en columna monto_descuento
50
Proporción de valores nulos en columna monto_descuento
0.00199203187250996
_____________________________________
Cantidad de valores nulos en columna monto_total
0
Proporción de valores nulos en columna monto_total
0.0
_____________________________________


Las 50 filas con nulos en cantidad, precio_unitario y monto_descuento no se imputan con mediana, promedio o cero:

cantidad es indispensable para medir unidades vendidas.

precio_unitario es indispensable para calcular ingresos, costos y margen.

monto_descuento podría interpretarse como 0 en otros contextos, pero aquí está vacío junto con cantidad y precio_unitario; por tanto, parece un bloque de información incompleta, no solamente un descuento ausente.

Aunque esas filas tengan monto_total, no es posible recuperar de manera única el producto, el precio o la cantidad. Por ejemplo, un total de 500 podría corresponder a una unidad de 500, dos unidades de 250, o una transacción con descuento.

Además, estos faltantes son muy pocos respecto a las 25.100 filas originales: 300 registros sin país representan aproximadamente 1,20%, y 50 registros con las tres variables numéricas faltantes aproximadamente 0,20%.

Excluirlos evita inventar información y conserva una muestra prácticamente completa.

In [ ]:
valores_nulos(catalog, col_num_catalog)

Cantidad de valores nulos en columna costo_unitario
0
Proporción de valores nulos en columna costo_unitario
0.0
_____________________________________


In [ ]:
valores_nulos(marketing, col_num_marketing)

Cantidad de valores nulos en columna gasto
0
Proporción de valores nulos en columna gasto
0.0
_____________________________________


In [ ]:
def explorar(df, columnas):
    for col in columnas:
        if df[col].dtype == 'int64' or df[col].dtype == 'float64':
            print(),
            print(f"Estadisticas descriptivas de {col}"),
            print("Mediana: ", df[col].median()),
            print(df[col].describe()),
            print('___________________________')

        elif df[col].dtype == object:
            print(),
            print(f"Estadisticas Descriptivas de {col}"),
            print(df[col].describe()),
            print(),
            print(f"Conteo de Nulos en {col}")
            print(df[col].isna().value_counts()),
            print(),
            print(f"Porcentaje de Nulos en {col}"),
            print(df[col].isna().value_counts(normalize=True)),
            print(),
            print(f"Participación Absoluta de cada categoría en {col}"),
            print(df[col].value_counts()),
            print(),
            print(f"Participación Porcentual de cada categoría en {col}"),
            print(df[col].value_counts(normalize=True))
            print("________________________________________")

        else:
            return print('no concuerdan los tipos de datos')

explorar(orders, col_num_orders)


Estadisticas descriptivas de cantidad
Mediana:  2.0
count    25050.000000
mean         7.092735
std        296.277003
min         -2.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64
___________________________

Estadisticas descriptivas de precio_unitario
Mediana:  258.71500000000003
count    25050.000000
mean       259.305549
std        138.726461
min         20.030000
25%        138.377500
50%        258.715000
75%        380.332500
max        499.960000
Name: precio_unitario, dtype: float64
___________________________

Estadisticas descriptivas de monto_descuento
Mediana:  0.0
count    25050.000000
mean         4.500798
std          5.223010
min          0.000000
25%          0.000000
50%          0.000000
75%         10.000000
max         15.000000
Name: monto_descuento, dtype: float64
___________________________

Estadisticas descriptivas de monto_total
Mediana:  341.75
count    2.510000e+04
mean     2.0

In [ ]:
# _____________________________________________________

In [ ]:
# LIMPIEZA Y ESTANDARIZACIÓN DEL VARIABLES NUMERICAS EN ORDERS

# Reemplazar valores negativos por la mediana de cantidad y monto total
orders.info()
# ============================================================
# REEMPLAZAR VALORES MENORES O IGUALES A CERO POR NaN
# ============================================================

# Crear una lista con las columnas numéricas donde
# los valores deben ser mayores que cero.
#
# No se incluye monto_descuento porque un descuento de 0 es válido.
columnas_a_limpiar = [
    'cantidad',
    'precio_unitario',
    'monto_total'
]

# Recorrer cada columna de la lista.
for columna in columnas_a_limpiar:

    # Buscar los valores menores o iguales a cero.
    #
    # <= 0 significa:
    # menor que cero O igual a cero.
    #
    # np.nan representa un valor faltante.
    #
    # loc permite modificar únicamente las filas que cumplen
    # la condición indicada.
    orders.loc[orders[columna] <= 0, columna] = np.nan

#Verificar cambios
explorar(orders, col_num_orders)
valores_nulos(orders, col_num_orders)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB

Estadisticas descriptivas de cantid

Ahora los resultados muestran que como habian antes 4 valores negativos al ser reemplazados por NaN quedan 54 y 4 en monto_total

In [ ]:
# ____________________________________________________

In [ ]:
# VERIFICACIÓN DE CONSISTENCIA DE MONTOS

In [ ]:
# ============================================================
# VERIFICAR CONSISTENCIA ENTRE CANTIDAD, PRECIO Y MONTO TOTAL
# ============================================================

# ------------------------------------------------------------
# 1. CALCULAR EL MONTO TOTAL ESPERADO
# ------------------------------------------------------------
# La fórmula es:
#
# cantidad * precio_unitario - monto_descuento
#
# El resultado se guarda en una nueva columna llamada
# 'monto_total_calculado'.
orders['monto_total_calculado'] = (
    orders['cantidad'] * orders['precio_unitario']
) - orders['monto_descuento']


# ------------------------------------------------------------
# 2. CALCULAR LA DIFERENCIA ENTRE AMBOS MONTOS
# ------------------------------------------------------------
# Restamos el monto informado en la base de datos menos
# el monto calculado con la fórmula.
#
# Si la diferencia es 0, el monto es consistente.
# Si la diferencia es distinta de 0, puede haber inconsistencia.
orders['diferencia_monto'] = (
    orders['monto_total'] - orders['monto_total_calculado']
)


# ------------------------------------------------------------
# 3. IDENTIFICAR LAS FILAS INCONSISTENTES
# ------------------------------------------------------------
# Identificar las filas que tienen todos los valores necesarios
# para verificar la fórmula:
# cantidad * precio_unitario - monto_descuento.
filas_completas = orders[
    [
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total'
    ]
].notna().all(axis=1)

# Redondear la diferencia a dos decimales.
# Con esto, 0.0100000000005 se convierte en 0.01.
diferencia_redondeada = orders['diferencia_monto'].round(2)

# Marcar como inconsistentes únicamente diferencias de 2 centavos
# o más, tanto positivas como negativas.
#
# 0.01 y -0.01 se consideran diferencias aceptables por redondeo.
filas_inconsistentes = (
    filas_completas
    & (
        (diferencia_redondeada >= 0.02)
        | (diferencia_redondeada <= -0.02)
    )
)

# ------------------------------------------------------------
# 4. CREAR UNA TABLA SOLO CON LAS INCONSISTENCIAS
# ------------------------------------------------------------
# Esta tabla muestra únicamente los pedidos donde la fórmula
# no coincide con el monto_total registrado.
inconsistencias_montos = orders[filas_inconsistentes]

# Seleccionar las columnas necesarias para revisar cada caso.
inconsistencias_montos = inconsistencias_montos[
    [
        'id_pedido',
        'cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total',
        'monto_total_calculado',
        'diferencia_monto'
    ]
]


# ------------------------------------------------------------
# 5. MOSTRAR EL REPORTE
# ------------------------------------------------------------
print('----- REPORTE DE CONSISTENCIA DE MONTOS -----')

# Mostrar la cantidad total de filas revisadas.
print('\nCantidad total de pedidos revisados:')
print(orders.shape[0])

# Mostrar la cantidad de filas inconsistentes.
print('\nCantidad de pedidos con montos inconsistentes:')
print(inconsistencias_montos.shape[0])

# Mostrar las primeras 10 inconsistencias para investigarlas.
print('\nPrimeras 10 filas con montos inconsistentes:')
print(inconsistencias_montos.head(10))

----- REPORTE DE CONSISTENCIA DE MONTOS -----

Cantidad total de pedidos revisados:
25100

Cantidad de pedidos con montos inconsistentes:
0

Primeras 10 filas con montos inconsistentes:
Empty DataFrame
Columns: [id_pedido, cantidad, precio_unitario, monto_descuento, monto_total, monto_total_calculado, diferencia_monto]
Index: []


NO EXISTEN INCONSISTENCIAS EN LOS MONTOS SI SE SONSIDERAN RANGOS MÁXIMO DE 1 CENTAVO LO CUAL PUEDE DEBERSE AL LENGUAJE DE PROGRAMACIÓN O CALCULO DE MULTIPLE DECIMALES POR ENDE SE PROCEDE CON EL SIGUIENTE PUNTO

In [ ]:
#_______________________________________________________

In [ ]:
# ELIMINACIÓN DE DUPLICADOS

En orders, la columna que debe utilizarse para identificar duplicados es id_pedido, porque representa el identificador del pedido. En el archivo original hay 25.100 filas, pero solo 25.000 id_pedido distintos: existen 100 IDs repetidos

In [ ]:
# ============================================================
# ELIMINAR DUPLICADOS DEL DATASET ORDERS
# ============================================================

# ------------------------------------------------------------
# 1. REVISAR DUPLICADOS COMPLETOS
# ------------------------------------------------------------
# duplicated() revisa si una fila completa aparece repetida.
#
# keep=False marca como True todas las apariciones de una fila
# repetida, incluida la primera.
filas_duplicadas_completas = orders.duplicated(
    keep=False
)

# Mostrar cuántas filas participan en duplicados completos.
print('Cantidad de filas duplicadas completamente:')
print(filas_duplicadas_completas.sum())

# Mostrar algunos ejemplos.
print('\nEjemplos de duplicados completos:')
print(orders[filas_duplicadas_completas].head())


# ------------------------------------------------------------
# 2. REVISAR DUPLICADOS POR ID_PEDIDO
# ------------------------------------------------------------
# id_pedido debe identificar un pedido individual.
#
# keep=False marca todos los registros cuyo id_pedido aparece
# más de una vez.
ids_pedido_duplicados = orders['id_pedido'].duplicated(
    keep=False
)

# Mostrar cuántas filas pertenecen a IDs repetidos.
print('\nCantidad de filas con id_pedido repetido:')
print(ids_pedido_duplicados.sum())

# Mostrar algunos ejemplos de IDs repetidos.
print('\nEjemplos de id_pedido repetidos:')
print(
    orders[ids_pedido_duplicados]
    .sort_values('id_pedido')
    .head(10)
)


# ------------------------------------------------------------
# 3. ELIMINAR DUPLICADOS COMPLETOS
# ------------------------------------------------------------
# drop_duplicates() elimina las filas que son idénticas
# en todas sus columnas.
#
# keep='first' conserva la primera aparición y elimina
# las repeticiones posteriores.
orders = orders.drop_duplicates(
    keep='first'
)


# ------------------------------------------------------------
# 4. ELIMINAR REPETICIONES DEL ID_PEDIDO
# ------------------------------------------------------------
# subset=['id_pedido'] indica que la revisión se realiza
# únicamente usando la columna id_pedido.
#
# keep='first' conserva el primer registro de cada pedido
# y elimina cualquier repetición posterior.
orders = orders.drop_duplicates(
    subset=['id_pedido'],
    keep='first'
)


# ------------------------------------------------------------
# 5. REINICIAR EL ÍNDICE
# ------------------------------------------------------------
# Después de eliminar filas, el índice puede conservar saltos.
#
# reset_index(drop=True) crea un índice nuevo:
# 0, 1, 2, 3, ...
#
# drop=True evita que el índice anterior se convierta
# en una columna adicional.
orders = orders.reset_index(
    drop=True
)


# ------------------------------------------------------------
# 6. COMPROBAR QUE YA NO HAY DUPLICADOS
# ------------------------------------------------------------
print('\n----- RESULTADO FINAL -----')
resumen_calidad(orders)


Cantidad de filas duplicadas completamente:
200

Ejemplos de duplicados completos:
       id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
734    order_734  user_6347        2025-05-02   Colombia     desktop   
812    order_812  user_1530        2025-03-31  Argentina     desktop   
974    order_974  user_3262        2025-05-04  Argentina     desktop   
1361  order_1361   user_507        2025-06-27     Mexico     desktop   
1735  order_1735   user_728        2025-05-13     Mexico      mobile   

     fuente_referencia       nombre_producto categoria_producto  cantidad  \
734            organic        Blender-XL-Red              Hogar       1.0   
812            organic  Tablet-Standard-64GB        Electronica       1.0   
974        paid_search        Blender-XL-Red              Hogar       2.0   
1361       paid_search    Laptop-Gaming-16GB        Electronica       2.0   
1735            social  Tablet-Standard-64GB        Electronica       2.0   

      precio_unitario

In [ ]:
# ______________________________________________

In [ ]:
# # ANALISIS DE COLUMNAS CATEGORICAS DE CADA DATAFRAME

In [ ]:
# Se imputan las categorias de producto a continuación dado que se cuenta con la info completa

# Crear una tabla de referencia: para cada producto, obtener su categoría
# set_index convierte 'nombre_producto' en el índice (la clave de búsqueda)
# ['categoria_producto'] extrae solo la columna de categorías
mapa_categoria = catalog.set_index('nombre_producto')['categoria_producto']

# Rellenar únicamente las categorías que están vacías (NaN)
# pedidos['nombre_producto'].map(mapa_categoria) busca la categoría de cada producto
# fillna reemplaza solo los valores faltantes, dejando intactos los que ya existen
orders['categoria_producto'] = orders['categoria_producto'].fillna(
    orders['nombre_producto'].map(mapa_categoria))

orders['categoria_producto'] = orders['categoria_producto'].replace({
    'Electrónica':'Electronica'})

# Se estandarizó la categoria Electronica

orders.info()
orders[orders['categoria_producto'].isna()]
orders['categoria_producto'].value_counts(dropna=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_pedido              25000 non-null  object        
 1   id_usuario             25000 non-null  object        
 2   fecha_hora_pedido      25000 non-null  datetime64[ns]
 3   pais                   24700 non-null  object        
 4   dispositivo            24980 non-null  object        
 5   fuente_referencia      24970 non-null  object        
 6   nombre_producto        24970 non-null  object        
 7   categoria_producto     24970 non-null  object        
 8   cantidad               24946 non-null  float64       
 9   precio_unitario        24950 non-null  float64       
 10  monto_descuento        24950 non-null  float64       
 11  monto_total            24996 non-null  float64       
 12  monto_total_calculado  24946 non-null  float64       
 13  d

Hogar          8355
Moda           8324
Electronica    8291
NaN              30
Name: categoria_producto, dtype: int64

Se limpiaron 50 de los 80 valores faltantes debido a que hay registro con nulos desde el producto haciendo imposible esa trazabilidad

In [ ]:
# Unificar la escritura de la categoría de tecnología
# En algunos registros aparece 'mexico' y en otros 'Mexico'
# replace cambia todos los nombres a un solo estandar dado
orders['pais'] = orders['pais'].replace({
    'mexico':'Mexico', 'colombia':'Colombia', 'argentina':'Argentina'})

orders['pais'].value_counts(dropna=False)

Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: pais, dtype: int64

Los 300 NaN de pais no se pueden reconstruir de forma fiable a partir de otras columnas: el país no está determinado por el producto el canal, el dispositivo ni el monto. Asignar el país más frecuente, por ejemplo, alteraría artificialmente los análisis por territorio.

In [ ]:
## Eliminar columnas creadas solo para revisar la consistencia
# de monto_total.
orders = orders.drop(
    columns=[
        'monto_total_calculado',
        'diferencia_monto'
    ]
)

## Definir las columnas que son obligatorias para analizar pedidos.
columnas_importantes_orders = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto',
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]



In [ ]:
# ============================================================
# FUNCIÓN PARA SEPARAR FILAS CON VALORES NULOS
# ============================================================

def separar_valores_nulos(df, columnas_importantes):

    # --------------------------------------------------------
    # 1. IDENTIFICAR FILAS QUE TIENEN ALGÚN VALOR NULO
    # --------------------------------------------------------
    # df[columnas_importantes] selecciona solo las columnas
    # que deben contener datos.
    #
    # isna() marca con True los valores NaN.
    #
    # any(axis=1) revisa cada fila:
    # devuelve True si al menos una de las columnas tiene NaN.
    filas_con_nulos = df[columnas_importantes].isna().any(axis=1)


    # --------------------------------------------------------
    # 2. CREAR EL DATASET AUDITABLE
    # --------------------------------------------------------
    # Guardar todas las filas que tienen por lo menos un NaN.
    #
    # copy() crea una copia independiente del DataFrame.
    datos_auditables_nulos = df[filas_con_nulos].copy()


    # --------------------------------------------------------
    # 3. CONSERVAR SOLO FILAS COMPLETAS
    # --------------------------------------------------------
    # El símbolo ~ significa "NO".
    #
    # Por eso, ~filas_con_nulos conserva las filas que no tienen
    # valores nulos en las columnas importantes.
    datos_sin_nulos = df[~filas_con_nulos].copy()


    # --------------------------------------------------------
    # 4. REINICIAR EL ÍNDICE DEL DATASET PRINCIPAL
    # --------------------------------------------------------
    # Crea una nueva numeración consecutiva: 0, 1, 2, 3, ...
    #
    # drop=True evita guardar el índice antiguo como columna.
    datos_sin_nulos = datos_sin_nulos.reset_index(drop=True)


    # --------------------------------------------------------
    # 5. MOSTRAR UN RESUMEN
    # --------------------------------------------------------
    print('----- RESUMEN DE SEPARACIÓN DE NULOS -----')

    print('\nFilas originales:')
    print(df.shape[0])

    print('\nFilas enviadas al dataset auditable:')
    print(datos_auditables_nulos.shape[0])

    print('\nFilas que permanecen en el dataset principal:')
    print(datos_sin_nulos.shape[0])

    print('\nValores nulos encontrados en el dataset auditable:')
    print(datos_auditables_nulos[columnas_importantes].isna().sum())


    # --------------------------------------------------------
    # 6. DEVOLVER LOS DOS DATASETS
    # --------------------------------------------------------
    # return entrega los dos DataFrames que creó la función.
    return datos_sin_nulos, datos_auditables_nulos



In [ ]:
# La función devuelve dos resultados:
#
# 1. orders: filas completas que se conservarán para el análisis.
# 2. orders_auditable_nulos: filas con NaN para trazabilidad.
orders, orders_auditable_nulos = separar_valores_nulos(
    orders,
    columnas_importantes_orders
)


## Definir la columna obligatoria para el análisis de marketing.
#
# canal es necesario para calcular gasto por canal.
columnas_importantes_marketing = [
    'canal'
]

# La función devuelve dos resultados:
#
# 1. marketing: registros que tienen canal identificado.
# 2. marketing_auditable_nulos: registros cuyo canal es NaN.
marketing, marketing_auditable_nulos = separar_valores_nulos(
    marketing,
    columnas_importantes_marketing
)

----- RESUMEN DE SEPARACIÓN DE NULOS -----

Filas originales:
25000

Filas enviadas al dataset auditable:
400

Filas que permanecen en el dataset principal:
24600

Valores nulos encontrados en el dataset auditable:
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad               54
precio_unitario        50
monto_descuento        50
monto_total             4
dtype: int64
----- RESUMEN DE SEPARACIÓN DE NULOS -----

Filas originales:
1620

Filas enviadas al dataset auditable:
101

Filas que permanecen en el dataset principal:
1519

Valores nulos encontrados en el dataset auditable:
canal    101
dtype: int64


In [ ]:
print('===== ORDERS =====')
resumen_calidad(orders)

print('\n===== ORDERS AUDITABLE NULOS =====')
resumen_calidad(orders_auditable_nulos)

print('\n===== MARKETING =====')
resumen_calidad(marketing)

print('\n===== MARKETING AUDITABLE NULOS =====')
resumen_calidad(marketing_auditable_nulos)

print('\n===== CATALOGO =====')
resumen_calidad(catalog)


===== ORDERS =====
----- INFORMACIÓN GENERAL -----

Número de filas: 24600
Número de columnas: 12

----- NOMBRES DE COLUMNAS -----
Index(['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo',
       'fuente_referencia', 'nombre_producto', 'categoria_producto',
       'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total'],
      dtype='object')

----- TIPOS DE DATOS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24600 entries, 0 to 24599
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24600 non-null  object        
 1   id_usuario          24600 non-null  object        
 2   fecha_hora_pedido   24600 non-null  datetime64[ns]
 3   pais                24600 non-null  object        
 4   dispositivo         24600 non-null  object        
 5   fuente_referencia   24600 non-null  object        
 6   nombre_producto     24600 non-null  o

In [ ]:
print('===== ORDERS =====')
resumen_calidad(orders)

print('\n===== ORDERS AUDITABLE NULOS =====')
resumen_calidad(orders_auditable_nulos)

print('\n===== MARKETING =====')
resumen_calidad(marketing)

print('\n===== MARKETING AUDITABLE NULOS =====')
resumen_calidad(marketing_auditable_nulos)

print('\n===== CATALOGO =====')
resumen_calidad(catalog)

===== ORDERS =====
----- INFORMACIÓN GENERAL -----

Número de filas: 24600
Número de columnas: 12

----- NOMBRES DE COLUMNAS -----
Index(['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo',
       'fuente_referencia', 'nombre_producto', 'categoria_producto',
       'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total'],
      dtype='object')

----- TIPOS DE DATOS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24600 entries, 0 to 24599
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24600 non-null  object        
 1   id_usuario          24600 non-null  object        
 2   fecha_hora_pedido   24600 non-null  datetime64[ns]
 3   pais                24600 non-null  object        
 4   dispositivo         24600 non-null  object        
 5   fuente_referencia   24600 non-null  object        
 6   nombre_producto     24600 non-null  o

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
orders_auditable_nulos.to_csv('orders_auditable_nulos.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)
marketing_auditable_nulos.to_csv('marketing_auditable_nulos.csv', index=False)

Se aplicó un modo de operación que consistía en la limpieza de datos, pero dado que muchos de esos datos requerían cálculos específicos, se optó por ponerlos en un archivo aparte de auditoría, de tal forma de que no se pierda su trazabilidad, se puedan completar los datos en algún momento y si se pueden llegar a completar unir con relativa facilidad. Como tal, todos los los datassets quedaron con sus respectivos formatos. Por lo que el dataset queda de manera uniforme y con todos las variables bien definidas. Sólo dataset de orders y marketing presentaban exceso de valores nulos por lo que se optó como, se mencionó, por moverlos hacia un dataset diferente. Existieron datos que sí que se podían extraer de la información de otros datasets o inclusive con el cálculo de otras columnas y así se trataron.

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders_clean`, `catalog_clean`, `marketing_clean`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# Cargar los datos
catalog_clean = pd.read_csv('catalog_clean.csv')
marketing_clean = pd.read_csv('marketing_clean.csv')
orders_clean = pd.read_csv('orders_clean.csv')


# ANÁLISIS DE RENTABILIDAD

# ¿Cuál es el ingreso total (revenue)?
ingreso_total = orders_clean['monto_total'].sum()

# =============================================================================
# ¿Cuál es el costo total?
# Unir catálogo con órdenes para obtener el costo de cada producto vendido
orders_clean_con_costo = orders_clean.merge(
    catalog_clean[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

# Calcular el costo de cada orden y sumar todos
costo_total = (orders_clean_con_costo['costo_unitario'] * orders_clean_con_costo['cantidad']).sum()

# =============================================================================
# ¿Cuánto se ha invertido en marketing?

gasto_marketing_total = marketing_clean['gasto'].sum()

# =============================================================================
# ¿El negocio es rentable? (calcular profit)
# =============================================================================
profit = ingreso_total - costo_total - gasto_marketing_total
margen_profit = (profit / ingreso_total) * 100

# =============================================================================
# Resultados
# =============================================================================
print("="*60)
print("RESULTADOS")
print("="*60)
print(f"Ingreso total (revenue):    ${ingreso_total:,.2f}")
print(f"Costo total:                ${costo_total:,.2f}")
print(f"Gasto en marketing:         ${gasto_marketing_total:,.2f}")
print(f"Profit:                     ${profit:,.2f}")
print(f"Margen de profit:           {margen_profit:.2f}%")
print("="*60)

if profit > 0:
    print("El negocio es rentable")
else:
    print("El negocio no es rentable")


RESULTADOS
Ingreso total (revenue):    $51,836,375.14
Costo total:                $43,078,678.80
Gasto en marketing:         $2,694,664.43
Profit:                     $6,063,031.91
Margen de profit:           11.70%
El negocio es rentable


In [ ]:
# =============================================================================
# PARTE 2: COMPORTAMIENTO DE VENTAS
# =============================================================================

# =============================================================================
# ¿Cuál es el ticket promedio por orden?
# Ticket promedio = monto total de todas las órdenes / número de órdenes
ticket_promedio = orders_clean['monto_total'].mean()

# =============================================================================
# ¿Cuál es la cantidad promedio de productos por orden?
# Sumamos todas las cantidades y dividimos entre el número de órdenes
cantidad_promedio = orders_clean['cantidad'].mean()

# =============================================================================
# ¿Cuál es el producto más vendido?
# Agrupamos por producto y sumamos las cantidades
ventas_por_producto = orders_clean.groupby('nombre_producto')['cantidad'].sum()
producto_mas_vendido = ventas_por_producto.idxmax()
cantidad_producto_mas_vendido = ventas_por_producto.max()

# =============================================================================
# ¿Cuánto se ha gastado en marketing por canal?
# Agrupamos por canal y sumamos los gastos
gasto_por_canal = marketing_clean.groupby('canal')['gasto'].sum().sort_values(ascending=False)

# =============================================================================
# Resultados
print("="*60)
print("PARTE 2: COMPORTAMIENTO DE VENTAS")
print("="*60)

print("\nTICKET PROMEDIO POR ORDEN")
print(f"   Ticket promedio: ${ticket_promedio:,.2f}")

print("\nCANTIDAD PROMEDIO DE PRODUCTOS POR ORDEN")
print(f"   Cantidad promedio: {cantidad_promedio:.2f} productos")


print("\nPRODUCTO MÁS VENDIDO")
print("top 5 productos más vendidos")
top_5_productos = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False).head(5)
print(top_5_productos)
print(" ____________________________ ")
print(f"Producto mas vendido: {producto_mas_vendido}")
print(f"Cantidad vendida: {cantidad_producto_mas_vendido:,.0f} unidades")


print("\nGASTO EN MARKETING POR CANAL")
for canal, gasto in gasto_por_canal.items():
    porcentaje = (gasto / gasto_por_canal.sum()) * 100         # gasto = el gasto de UN canal y gasto_por_canal.sum() = la suma de TODOS los canales
    print(f"   {canal}: ${gasto:,.2f} ({porcentaje:.1f}%)")


print("\n" + "="*60)

PARTE 2: COMPORTAMIENTO DE VENTAS

TICKET PROMEDIO POR ORDEN
   Ticket promedio: $2,107.17

CANTIDAD PROMEDIO DE PRODUCTOS POR ORDEN
   Cantidad promedio: 7.20 productos

PRODUCTO MÁS VENDIDO
top 5 productos más vendidos
nombre_producto
Laptop-Gaming-16GB    144160.0
Vacuum-Pro-Black        6203.0
Jacket-Winter-M         6185.0
Blender-XL-Red          6184.0
Sneakers-Urban-42       6085.0
Name: cantidad, dtype: float64
 ____________________________ 
Producto mas vendido: Laptop-Gaming-16GB
Cantidad vendida: 144,160 unidades

GASTO EN MARKETING POR CANAL
   social: $918,043.21 (34.1%)
   organic: $913,533.01 (33.9%)
   paid_search: $863,088.21 (32.0%)



En un Series de pandas, el índice puede ser cualquier cosa, no necesariamente números. El índice aquí son los nombres de los canales ('paid_search', 'organic', 'social'), NO los números 0, 1, 2. pandas agrupa por la columna 'canal'. Los valores únicos de esa columna se convierten en el índice del resultado. El índice deja de ser 0, 1, 2... y ahora es 'paid_search', 'organic', 'social'.

.index[i]      # obtiene el índice
.iloc[i]       # obtiene el valor

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

# PARTE 1: TOTALES DEL FUNNEL
# =================================

query_totals = '''

-- Seleccionamos el nombre de cada tipo de evento
SELECT nombre_evento,

-- Contamos usuarios únicos que realizaron cada evento
COUNT(DISTINCT id_usuario) AS usuarios

-- Indicamos que los datos se obtienen de la tabla events
FROM events

-- Agrupamos los resultados por tipo de evento
GROUP BY nombre_evento

-- Ordenamos las etapas por cantidad de usuarios, de mayor a menor
ORDER BY usuarios DESC;

'''

# Ejecutamos la consulta SQL y guardamos el resultado en un DataFrame
totals = pd.read_sql(query_totals, con=engine)

# Mostramos la tabla con los totales de usuarios por etapa
totals

,nombre_evento,usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


Tal parece ser que entre cada uno de los canales del Funnel no hay un drop-off tan drastico. Por lo que se procede al análisis de conversión para ver en términos porcentuales que tan Drásticas son los cambios de una interaccion a otra.

In [ ]:
# PARTE 2: ANALISIS DE CONVERSION Y DROP-OFF
# =================================

query_conversion = '''

-- Contamos los usuarios únicos de cada evento
WITH cte_eventos AS (

    -- Agrupamos por el nombre de cada evento
    SELECT
        nombre_evento,

        -- Contamos usuarios únicos que realizaron cada evento
        COUNT(DISTINCT id_usuario) AS usuarios

    -- Usamos la tabla events
    FROM events

    -- Creamos un total para cada nombre de evento
    GROUP BY nombre_evento
)

-- Calculamos las tasas entre las etapas del funnel
SELECT

    -- Conversión de first_visit a add_to_cart
    ROUND(
        (
            SELECT usuarios
            FROM cte_eventos
            WHERE nombre_evento = 'add_to_cart'
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'first_visit'
            ),
            0
        ),
        2
    ) AS conversion_first_visit_add_to_cart,

    -- Drop-off de first_visit a add_to_cart
    ROUND(
        (
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'first_visit'
            )
            -
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_to_cart'
            )
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'first_visit'
            ),
            0
        ),
        2
    ) AS dropoff_first_visit_add_to_cart,

    -- Conversión de add_to_cart a select_item
    ROUND(
        (
            SELECT usuarios
            FROM cte_eventos
            WHERE nombre_evento = 'select_item'
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_to_cart'
            ),
            0
        ),
        2
    ) AS conversion_add_to_cart_select_item,

    -- Drop-off de add_to_cart a select_item
    ROUND(
        (
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_to_cart'
            )
            -
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'select_item'
            )
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_to_cart'
            ),
            0
        ),
        2
    ) AS dropoff_add_to_cart_select_item,

    -- Conversión de select_item a begin_checkout
    ROUND(
        (
            SELECT usuarios
            FROM cte_eventos
            WHERE nombre_evento = 'begin_checkout'
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'select_item'
            ),
            0
        ),
        2
    ) AS conversion_select_item_begin_checkout,

    -- Drop-off de select_item a begin_checkout
    ROUND(
        (
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'select_item'
            )
            -
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'begin_checkout'
            )
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'select_item'
            ),
            0
        ),
        2
    ) AS dropoff_select_item_begin_checkout,

    -- Conversión de begin_checkout a add_payment_info
    ROUND(
        (
            SELECT usuarios
            FROM cte_eventos
            WHERE nombre_evento = 'add_payment_info'
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'begin_checkout'
            ),
            0
        ),
        2
    ) AS conversion_begin_checkout_add_payment_info,

    -- Drop-off de begin_checkout a add_payment_info
    ROUND(
        (
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'begin_checkout'
            )
            -
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_payment_info'
            )
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'begin_checkout'
            ),
            0
        ),
        2
    ) AS dropoff_begin_checkout_add_payment_info,

    -- Conversión de add_payment_info a purchase
    ROUND(
        (
            SELECT usuarios
            FROM cte_eventos
            WHERE nombre_evento = 'purchase'
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_payment_info'
            ),
            0
        ),
        2
    ) AS conversion_add_payment_info_purchase,

    -- Drop-off de add_payment_info a purchase
    ROUND(
        (
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_payment_info'
            )
            -
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'purchase'
            )
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'add_payment_info'
            ),
            0
        ),
        2
    ) AS dropoff_add_payment_info_purchase,

    -- Tasa de conversión final:
    -- usuarios del último paso / usuarios del primer paso × 100
    ROUND(
        (
            SELECT usuarios
            FROM cte_eventos
            WHERE nombre_evento = 'purchase'
        )::numeric
        * 100
        /
        NULLIF(
            (
                SELECT usuarios
                FROM cte_eventos
                WHERE nombre_evento = 'first_visit'
            ),
            0
        ),
        2
    ) AS conversion_final

'''

# Ejecutamos la consulta SQL
conversion = pd.read_sql(query_conversion, con=engine)

# Mostramos únicamente los porcentajes calculados
conversion

,conversion_first_visit_add_to_cart,dropoff_first_visit_add_to_cart,conversion_add_to_cart_select_item,dropoff_add_to_cart_select_item,conversion_select_item_begin_checkout,dropoff_select_item_begin_checkout,conversion_begin_checkout_add_payment_info,dropoff_begin_checkout_add_payment_info,conversion_add_payment_info_purchase,dropoff_add_payment_info_purchase,conversion_final
0,97.92,2.08,99.32,0.68,95.07,4.93,86.71,13.29,99.84,0.16,80.04


# Se calcula la tasa de conversión entre cada paso del funnel

La tasa de conversión entre cada una de las interacciones dentro de la plataforma gira en torno al 90%. Por lo que en sí misma no representa una fuga masiva en algún. Que el fonel Esto en sí mismo, mi indicador de que no hay como tal. Asuntos graves que resolver ni hay fricciones graves para el usuario en la plataforma


# Se identifica en qué etapa se pierde la mayor cantidad de usuarios

Lo anterior no quiere decir que si se puedan identificar fugas dentro de la misma plataforma, siendo el tema de empezar el checkout y agregar el método de pago donde más hay fuga de clientes, tal parece ser que puede estar habiendo una fricción ahí, a lo mejor el cliente no se siente cómodo como funciona el sistema de pagos o la información no es clara y deserta la compra por lo que se recomienda revisar temas de promociones. También se identifica un pequeño drop off entre cuando seleccionan un ítem y realmente empiezan el tema de facturación. Por lo visto, los carritos que dan con objetos dentro, pero nunca terminan dentro De tan siquiera que el cliente se plantee pagarlos, es decir, no hay suficiente intensidad ni tampoco interés como los clientes están viendo el producto y al final acaban de acertando de pagar. De pagarlos o están viendo también de pronto que la competencia a último momento. Y les está ofreciendo lo mejor

# ¿Cuál es la tasa de conversión final?

Con lo anterior planteado, solo el 80% de los clientes que empiezan proyecto de la plataforma acaban comprando realmente de por si no es un mal indicador, pero sí deberían haber mejoras en el sistema de pagos por si de pronto el cliente requiere que se le convenza de no desertar su compra, o si hay algún lío con las promociones, Información o métodos de pago

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# RETENCIÓN SEMANAL POR COHORTES
# ===============================

query_cohort_retention_final = '''

-- PASO 1: Identificamos la cohorte semanal de cada usuario
WITH cohortes AS (

    -- Seleccionamos el identificador de cada usuario
    SELECT
        u.id_usuario,

        -- Convertimos la fecha de registro al inicio de su semana
        DATE_TRUNC(
            'week',
            CAST(u.fecha_registro AS DATE)
        ) AS cohorte_semana

    -- Utilizamos la tabla users
    FROM users AS u
),

-- PASO 2: Relacionamos cada usuario con su actividad semanal
actividad_semanal AS (

    -- Seleccionamos la cohorte semanal del usuario
    SELECT
        c.cohorte_semana,

        -- Conservamos el identificador del usuario
        c.id_usuario,

        -- Tomamos la semana transcurrida desde el registro
        ua.dias_despues_registro,

        -- Conservamos si el usuario estuvo activo
        ua.activo

    -- Comenzamos desde las cohortes
    FROM cohortes AS c

    -- Unimos la actividad con el usuario correspondiente
    LEFT JOIN user_activity AS ua
        ON c.id_usuario = ua.id_usuario

),

-- PASO 3: Calculamos los usuarios iniciales y los retenidos
retencion AS (

    -- Agrupamos los resultados por cohorte semanal
    SELECT
        cohorte_semana,

        -- Contamos los usuarios iniciales de cada cohorte
        COUNT(DISTINCT id_usuario) AS clientes_iniciales,

        -- Contamos usuarios activos durante la semana 1
        COUNT(DISTINCT CASE
            WHEN dias_despues_registro BETWEEN 1 AND 7
                 AND activo = 1
            THEN id_usuario
        END) AS retenido_w1,

        -- Contamos usuarios activos durante la semana 2
        COUNT(DISTINCT CASE
            WHEN dias_despues_registro BETWEEN 8 AND 14
                 AND activo = 1
            THEN id_usuario
        END) AS retenido_w2,

        -- Contamos usuarios activos durante la semana 3
        COUNT(DISTINCT CASE
            WHEN dias_despues_registro BETWEEN 15 AND 21
                 AND activo = 1
            THEN id_usuario
        END) AS retenido_w3,

        -- Contamos usuarios activos durante la semana 4
        COUNT(DISTINCT CASE
            WHEN dias_despues_registro BETWEEN 22 AND 28
                 AND activo = 1
            THEN id_usuario
        END) AS retenido_w4


    -- Utilizamos la actividad semanal
    FROM actividad_semanal

    -- Agrupamos por cohorte semanal
    GROUP BY cohorte_semana
)

-- PASO 4: Mostramos la tabla final de retención
SELECT

    -- Mostramos la semana de registro de la cohorte
    TO_CHAR(cohorte_semana, 'YYYY-MM-DD') AS cohorte,

    -- Mostramos los usuarios iniciales
    clientes_iniciales,

    -- Mostramos los usuarios retenidos en cada semana
    retenido_w1,
    retenido_w2,
    retenido_w3,
    retenido_w4,


    -- Calculamos el porcentaje de retención de la semana 1
    ROUND(
        retenido_w1::numeric
        / NULLIF(clientes_iniciales, 0)
        * 100,
        2
    ) AS retencion_w1_pct,

    -- Calculamos el porcentaje de retención de la semana 2
    ROUND(
        retenido_w2::numeric
        / NULLIF(clientes_iniciales, 0)
        * 100,
        2
    ) AS retencion_w2_pct,

    -- Calculamos el porcentaje de retención de la semana 3
    ROUND(
        retenido_w3::numeric
        / NULLIF(clientes_iniciales, 0)
        * 100,
        2
    ) AS retencion_w3_pct,

    -- Calculamos el porcentaje de retención de la semana 4
    ROUND(
        retenido_w4::numeric
        / NULLIF(clientes_iniciales, 0)
        * 100,
        2
    ) AS retencion_w4_pct

-- Utilizamos la tabla de retención
FROM retencion

-- Ordenamos las cohortes desde la más antigua
ORDER BY cohorte_semana;

'''

# Ejecutamos la consulta SQL
cohorte_final = pd.read_sql(
    query_cohort_retention_final,
    con=engine
)

# Mostramos la tabla final de retención semanal
cohorte_final


,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,retenido_w4,retencion_w1_pct,retencion_w2_pct,retencion_w3_pct,retencion_w4_pct
0,2024-12-30,236,99,91,95,97,41.95,38.56,40.25,41.10
1,2025-01-06,351,145,157,148,138,41.31,44.73,42.17,39.32
2,2025-01-13,362,158,138,152,148,43.65,38.12,41.99,40.88
3,2025-01-20,394,174,156,157,159,44.16,39.59,39.85,40.36
4,2025-01-27,373,155,177,149,164,41.55,47.45,39.95,43.97
5,2025-02-03,405,164,177,184,168,40.49,43.70,45.43,41.48
6,2025-02-10,364,153,145,164,137,42.03,39.84,45.05,37.64
7,2025-02-17,338,159,136,135,138,47.04,40.24,39.94,40.83
8,2025-02-24,353,149,148,151,145,42.21,41.93,42.78,41.08
9,2025-03-03,363,144,158,146,147,39.67,43.53,40.22,40.50


---

Cerca de solo el 40% de los usuarios tiende a quedarse y estabilizarse a lo largo de las semanas, lo cual puede indicar un patrón de consumo bastante peculiar y pues sería ideal investigar por qué se mantiene específicamente así, qué es lo que el otro 60% de los usuarios no les motiva volver otra vez la plataforma y que podría hacerse para que. No se pierdan todavía más usuarios después de esas semanas.

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversión de compra despues de la modificación de la UI es igual o al menos muy similar
   - **H₁ (Hipótesis alternativa):** La tasa de conversión de compra despues de la modificación de la UI es diferente
   
**Test estadístico:** Z-test de proporciones

**Nivel de significancia alpha:** 5% (0.05)

In [ ]:
experiment_checkout_ui = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")
experiment_checkout_ui.head(5)


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
# Número de usuarios convertidos por página
conversiones = experiment_checkout_ui.groupby('variante')['convirtio'].sum()
totales = experiment_checkout_ui.groupby('variante')['convirtio'].count()

# Total de usuarios por página
conteos = [conversiones['control'], conversiones['tratamiento']]
num_observaciones = [totales['control'], totales['tratamiento']]

print("Usuarios convertidos por UI:\n", conteos)
print("\nTotal de usuarios por UI:\n", num_observaciones)

Usuarios convertidos por UI:
 [779, 820]

Total de usuarios por UI:
 [4965, 5035]


In [ ]:
# Aplicar prueba
from statsmodels.stats.proportion import proportions_ztest


# Ejecutar el z-test para visualizar resultados
z_stat, p_value = proportions_ztest(conteos, num_observaciones)
print(f"Estadístico z: {z_stat:.5f}")
print(f"Valor P: {p_value:.7f}")

alpha = 0.05  # umbral de significancia
if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.")

#---------------------------------------------------------------------------------------------
# Obtener tasa o porcentaje de éxito por grupo
tasa_A = conteos[0] / num_observaciones[0]
tasa_B = conteos[1] / num_observaciones[1]

print(f"\nTasa UI Control: {tasa_A:.2%}")
print(f"Tasa UI Tratamiento: {tasa_B:.2%}")
print(f"Diferencia: {tasa_B - tasa_A:.2%}")


Estadístico z: -0.81328
Valor P: 0.4160585
No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.

Tasa UI Control: 15.69%
Tasa UI Tratamiento: 16.29%
Diferencia: 0.60%


---

La pequeña diferencia entre los usuarios que interactuaron con la nueva UI en el checkout puede estar deviendose al azar, por lo que no podemos afirmar que sea mejor la nueva implementación

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

Encuentro una contradicción en el enfoque de multiples opciones del dashboard 1 con el detalle del dashboard 2 que pretende ser a donde lleve el drill-through, ya que como tal yo seleccioné que el grafico de barras sea por cateogoría por temas estetico y apiñamientos que dificultan la lectura del stakeholder pero luego el drill-trough objeta que debe ser si o si con un "producto" y encima "todos los pedidos relacionados" lo cual da pie a pensar que incluso la tabla que en el propio dashboard 2 pide debe incluir una columna con id_pedido si se quieren ver todos los pedidos. Considero que esto último puede estar siendo error de léxico utilizado dado que la misma IA dice que antes, cuando se pedía "Cantidad promedio de productos por orden" el "por orden", decía la IA, se interpretaba como "...cada vez que un cliente hace clic en "comprar" y confirma su carrito, eso genera una orden/pedido, identificado en (la) tabla por id_pedido". Por ende se pone una tabla adicional en el primer dashboard para compensar.

Respecto a la resolución de las preguntas objetivo del proyecto, en general se pueden confiar en los datos, dado que hay una inmensa cantidad de datos que sí tienen consistencia comprobada. Como tal el negocio sigue siendo rentable, aunque su ritmo está bajando por el modelo de negocio de suscripción con sus clientes, donde muchos de los productos se tienen que subvencionar a costo de suscripción (pese a no estar los datos, se puede inferir fácilmente). También existe una gran cantidad de clientes que compran al por mayor, convendría revisar bien esa parte, ya que como tal, Rappi no es una empresa que venda o distribuya productos al por mayor, sino que estos son más que todo minoristas, es un Marketplace donde la mayoría de tiendas son para consumidor final o clientes minoristas, por lo que puede ser un error de la plataforma desde el cual se están aprovechando algunos de los clientes para obtener productos a precios muy reducidos solo por la suscripción que poseen. Convendría revisar eso antes de que se vuelva un problema para la rentabilidad del negro mediado plazo.

Por otras parte, los usuarios se están perdiendo en el checkout; al momento de poner su método de pago, puede ser debido a falta de información ya sea bancaria o también, ya sea por costos asociados que después están viendo los clientes en la plataforma al final de la compra y que no logran comprender del todo como tarifas de servicio, tarifas de Del MarketPlacen tarifas de envíon, entre otras. Solo el 40% de los usuarios que compran por primera vez tienden a quedarse después de hasta 3 semanas; en general tienen un un tiempo de respuesta para volver bastante decente, al menos en el primer mes de estudio. Respecto a las compras dentro del Marketplace y los productos que como tal se pueden registrar en las en las bases de datos iniciales, los clientes tienden a volver a partir del cuarto mes. Hay un repunte bastante interesante que convendría entender y estudiar más a fondo con otro tipo de datos para comprender por qué es que están volviendo, qué es lo que lo que está motivando, ya sea a el precio de la membresía, los descuentos, temporadas y demás datos que de momento no están presentes, pero si como comportamiento recurrente de los clientes que compran inicialmente.

Respecto a los cambios, si generan impacto como tal, no hay evidencia estadísticamente significativa de que el cambio la interfaz de usuario represente un cambio significativo. Esto quiere decir que debería tantearse con otro tipo de interfaz de usuario que sean más llamativas que presenten información en tiempo real al cliente para que sienta confianza al momento de comprar y que asimismo los estadísticos reflejen ese comportamiento entre la preferencia de una interfaz o la otra. Como tal, el negocio presenta fortalezas en el tema de facturación y la masividad a la que están llegando con su público por ende, se puede usar y aprovechar para menores costos de marketing a mediano plazo y reestructuración de los beneficios otorgados por la suscripción los cuales deben de ser evaluados más a profundidad dada la cantidad de personas que están comprando a un costo mucho menor del que realmente está afrontando la empresa.

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive
# https://drive.google.com/file/d/1GqinI_xLLF7Gvq5RDdccaofIdF7mrsQW/view?usp=sharing